# 📚 Smart Research Assistant — Pipeline Walkthrough

A module-by-module tour of the RAG pipeline, in data-flow order: **ingest → clean → chunk → embed → index → retrieve → generate → evaluate → visualize**.

To keep memory modest, a single `RAGPipeline` is built once and its components (embedding model, retriever, generator) are reused throughout. Run from the project root.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import src.utils.env  # HuggingFace/TF env fix
from src.utils.config import AppConfig
cfg = AppConfig.load()
print('Working dir:', os.getcwd())
print('Embedding:', cfg.get('embeddings.active'), '| Vector store:', cfg.get('vector_store.active'))

## 1. Ingestion — PDF / TXT / DOCX / MD into uniform `Document`s

In [ ]:
from src.ingestion.document_loader import DocumentIngestor
docs = DocumentIngestor().ingest_directory('data/raw')
for d in docs:
    print(f'{d.doc_type:5s} {d.metadata.get("filename"):25s} {d.num_words:5d} words')

## 2. Preprocessing — normalize & clean (page boundaries preserved)

In [ ]:
from src.preprocessing.cleaner import DocumentPreprocessor
clean_docs = DocumentPreprocessor().process_many(docs)
print(clean_docs[0].content[:250])

## 3. Chunking — overlapping, page-attributed chunks

In [ ]:
from src.chunking.document_chunker import DocumentChunker
chunks = DocumentChunker().chunk_documents(clean_docs)
print('Total chunks:', len(chunks))
print('Metadata:', chunks[0].to_metadata())

## 4. Build the pipeline (one embedding-model load)
Indexing runs ingest → clean → chunk → embed → store internally.

In [ ]:
from src.pipeline import RAGPipeline
pipe = RAGPipeline(cfg, auto_load=False)
stats = pipe.index_directory('data/raw')
print(f'Indexed {stats.documents} docs -> {stats.chunks} chunks in {stats.elapsed_seconds:.2f}s')
print('Per-stage seconds:', {k: round(v,3) for k,v in stats.per_stage_seconds.items()})

## 5. Embeddings — reuse the pipeline's model

In [ ]:
model = pipe.embedding_model
v = model.embed_texts(['maternity leave', 'data encryption'])
print('dim:', model.dimension, '| shape:', v.shape, '| cosine:', float(v[0] @ v[1]))

## 6. Retrieval — top-K with adaptive relevance filtering

In [ ]:
for rc in pipe.retrieve('What is the maternity leave policy?', top_k=3):
    print(f'[{rc.rank}] {rc.score:.3f}  {rc.content[:70]}')

## 7. Generation — grounded answer + sources (loads local LLM once)

In [ ]:
resp = pipe.query('How must customer data be encrypted at rest?')
print('ANSWER:', resp.answer)
print('SOURCES:', resp.sources)
print('TIMINGS:', {k: round(v,2) for k,v in resp.timings.items()})

## 8. Evaluation — built-in metrics (or RAGAS with an API key)

In [ ]:
from src.evaluation.evaluator import RAGEvaluator
ev = RAGEvaluator(pipe.embedding_model, pipe.config)
report = ev.evaluate_pipeline(
    pipe,
    ['What is the maternity leave policy?', 'How is data encrypted at rest?'],
    ['26 weeks of paid maternity leave.', 'Using AES-256 at rest.'])
print('Method:', report.method)
print({k: round(v,3) for k,v in report.aggregate.items() if v==v})

## 9. Visualization — pure functions returning Plotly figures (PCA shown)

In [ ]:
import src.visualization as viz
emb = pipe.embedding_model.embed_texts([c.content for c in pipe.chunks])
labels = [c.metadata.get('filename', c.source) for c in pipe.chunks]
fig_space = viz.embedding_scatter(emb, labels, method='pca')
fig_radar = viz.metrics_radar(report.aggregate)
fig_chunks = viz.chunk_length_distribution(pipe.chunks)
print('Built figures:', [type(f).__name__ for f in (fig_space, fig_radar, fig_chunks)])
fig_radar  # renders inline in Jupyter